# DyT vs LayerNorm in SegFormer-B0 (ADE20K)

Companion notebook for the 6.s058 final paper (repo: [zhizhen-kyle-luo/6s058-final-project](https://github.com/zhizhen-kyle-luo/6s058-final-project)). Trains two SegFormer-B0 variants on an ADE20K subset:
- **LN baseline**: stock SegFormer-B0
- **DyT-full**: every backbone LayerNorm replaced with Dynamic Tanh (Zhu et al. 2025)

Pipeline: install -> clone repo (+ optional Drive) -> CPU smoke -> GPU smoke -> 500-iter quick runs -> 5000-iter full runs -> figures.

Designed for Colab Pro (T4 / L4 / A100). With Drive mounted, outputs persist across runtime resets.

**v1 layout note.** All logic is self-contained in this notebook so the project can run end-to-end on Colab with one click. Splitting into a `code/*.py` package is a future cleanup pass once results are in (relevant if this work goes to arXiv).

## 1. Install pinned deps + print versions

In [ ]:
!pip install -q "transformers==4.46.3" "datasets==3.1.0" "evaluate==0.4.3" "accelerate==1.1.1" "timm==1.0.11" 2>&1 | tail -n 3


In [ ]:
import torch, transformers, datasets, accelerate, timm
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("accelerate  ", accelerate.__version__)
print("timm        ", timm.__version__)
print("cuda        ", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


## 2. Clone repo + (optional) Drive for output persistence

This notebook is self-bootstrapping. On Colab it clones the public repo `zhizhen-kyle-luo/6s058-final-project` and runs from there — no manual upload needed.

Drive mount is optional but recommended: if available, `runs/` and `figures/` write to your Drive so checkpoints survive runtime resets. Without Drive, outputs land under `/content/` and disappear on reset.

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/zhizhen-kyle-luo/6s058-final-project.git"
REPO_DIR = pathlib.Path("/content/6s058-final-project")

try:
    from google.colab import drive
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    # 1. clone (or fast-forward pull) the repo
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    FINAL_PAPER_DIR = REPO_DIR

    # 2. optional Drive mount for output persistence across runtime resets
    USE_DRIVE_FOR_OUTPUTS = True
    if USE_DRIVE_FOR_OUTPUTS:
        try:
            drive.mount("/content/drive", force_remount=False)
            DRIVE_OUTPUTS = pathlib.Path("/content/drive/MyDrive/6s058-final-project")
            DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
            RUNS_DIR    = DRIVE_OUTPUTS / "runs"
            FIGURES_DIR = DRIVE_OUTPUTS / "figures"
            print(f"outputs persisting to Drive: {DRIVE_OUTPUTS}")
        except Exception as e:
            print(f"Drive mount failed ({e}); using ephemeral /content")
            RUNS_DIR    = FINAL_PAPER_DIR / "runs"
            FIGURES_DIR = FINAL_PAPER_DIR / "figures"
    else:
        RUNS_DIR    = FINAL_PAPER_DIR / "runs"
        FIGURES_DIR = FINAL_PAPER_DIR / "figures"
else:
    # local fallback (e.g. M3 quick check). Walk up to find final_paper/.
    here = pathlib.Path.cwd()
    candidates = [here, *here.parents]
    FINAL_PAPER_DIR = next((p / "final_paper" for p in candidates if (p / "final_paper").exists()), None)
    if FINAL_PAPER_DIR is None:
        FINAL_PAPER_DIR = pathlib.Path("./final_paper")
        FINAL_PAPER_DIR.mkdir(parents=True, exist_ok=True)
    RUNS_DIR    = FINAL_PAPER_DIR / "runs"
    FIGURES_DIR = FINAL_PAPER_DIR / "figures"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("FINAL_PAPER_DIR =", FINAL_PAPER_DIR)
print("RUNS_DIR        =", RUNS_DIR)
print("FIGURES_DIR     =", FIGURES_DIR)
!nvidia-smi -L 2>/dev/null || echo "no nvidia-smi"


## 3. Config

Defaults are conservative for a T4 (`batch=4`, `crop=512`, AMP on). Bump `batch` to 8–16 on L4/A100.

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    variant: str = "ln"            # "ln" | "dyt_full" | "dyt_attn" | "dyt_ffn"
    iters: int = 5000
    batch: int = 4                 # T4-safe; raise to 8 on L4, 16 on A100
    crop: int = 512
    lr: float = 6e-5
    weight_decay: float = 0.01
    warmup_iters: int = 200
    eval_every: int = 500
    subset_train: int = 2000
    subset_val: int = 500
    seed: int = 42
    pretrained: str = "nvidia/mit-b0"
    num_classes: int = 150
    ignore_index: int = 255
    dyt_alpha_init: float = 0.5
    use_amp: bool = True           # mixed precision; required to fit T4 at crop=512

print(Config())


## 4. DyT module + LayerNorm replacement

`DyT(x) = gamma * tanh(alpha * x) + beta`, learnable scalar `alpha` and per-channel `gamma`, `beta`.

The module supports **both** `(..., C)` (channel-last; how SegFormer's MiT applies LN) and `(B, C, H, W)` tensors. For NCHW inputs, `gamma` and `beta` are broadcast over `H, W` via reshape. SegFormer backbone replacement only hits `(..., C)` sites, but full NCHW support keeps the module general.

`replace_ln_with_dyt` walks the SegFormer backbone, swaps every `nn.LayerNorm` in place, and **prints the replaced module names** for reproducibility.

In [ ]:
import torch
import torch.nn as nn

class DyT(nn.Module):
    def __init__(self, normalized_shape, alpha_init: float = 0.5):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.alpha = nn.Parameter(torch.tensor(float(alpha_init)))
        self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        self.beta  = nn.Parameter(torch.zeros(self.normalized_shape))

    def forward(self, x):
        # NCHW first to disambiguate (B,C,H,C): treat 4D w/ x.shape[1]==C as NCHW
        if x.dim() == 4 and len(self.normalized_shape) == 1 and x.shape[1] == self.normalized_shape[0]:
            g = self.gamma.view(1, -1, 1, 1)
            b = self.beta.view(1, -1, 1, 1)
            return g * torch.tanh(self.alpha * x) + b
        # channel-last (..., C): gamma/beta broadcast on the last dim(s)
        if x.shape[-len(self.normalized_shape):] == self.normalized_shape:
            return self.gamma * torch.tanh(self.alpha * x) + self.beta
        raise RuntimeError(
            f"DyT shape mismatch: input {tuple(x.shape)} vs normalized_shape {self.normalized_shape}"
        )

    def extra_repr(self):
        return f"shape={self.normalized_shape}, alpha=learnable"


def _set_module(root: nn.Module, dotted: str, new: nn.Module):
    parts = dotted.split(".")
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def replace_ln_with_dyt(model: nn.Module, scope: str = "all", alpha_init: float = 0.5, verbose: bool = True):
    backbone_prefix = "segformer.encoder"
    replaced = []
    for name, module in list(model.named_modules()):
        if not isinstance(module, nn.LayerNorm):
            continue
        if not name.startswith(backbone_prefix):
            continue
        if scope == "attention" and "layer_norm_1" not in name:
            continue
        if scope == "ffn" and "layer_norm_2" not in name:
            continue
        shape = module.normalized_shape
        params = list(module.parameters())
        device = params[0].device if params else torch.device("cpu")
        dtype = params[0].dtype if params else torch.float32
        new = DyT(shape, alpha_init=alpha_init).to(device=device, dtype=dtype)
        _set_module(model, name, new)
        replaced.append(name)
    if verbose:
        print(f"[replace_ln_with_dyt] scope={scope} replaced {len(replaced)} LayerNorm modules:")
        for n in replaced:
            print("   ", n)
    return replaced


## 5. Saturation hooks

In [ ]:
from collections import defaultdict

class SaturationTracker:
    def __init__(self, model: nn.Module, threshold: float = 0.95):
        self.threshold = threshold
        self.values = defaultdict(list)
        self.handles = []
        for name, m in model.named_modules():
            if isinstance(m, DyT):
                self.handles.append(m.register_forward_hook(self._make_hook(name)))

    def _make_hook(self, name):
        def hook(module, inp, out):
            with torch.no_grad():
                pre = torch.tanh(module.alpha * inp[0])
                frac = (pre.abs() > self.threshold).float().mean().item()
            self.values[name].append(frac)
        return hook

    def summary(self):
        return {k: float(sum(v) / max(len(v), 1)) for k, v in self.values.items()}

    def close(self):
        for h in self.handles:
            h.remove()


## 6. Data: ADE20K via `scene_parse_150`

ADE20K convention: 150 foreground classes. The HF `scene_parse_150` dataset returns label maps with values `0..150`, where `0` is "background / unlabeled" and `1..150` are the foreground classes. We remap `0 -> 255 (ignore)` and `1..150 -> 0..149`.

A 2k-train / 500-val random subset is frozen by seed so LN and DyT see the same images.

In [ ]:
import random
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import functional as TF

IM_MEAN = (0.485, 0.456, 0.406)
IM_STD  = (0.229, 0.224, 0.225)


def _seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ADE20KSubset(Dataset):
    def __init__(self, hf_split, indices, crop: int, train: bool):
        self.ds = hf_split
        self.indices = indices
        self.crop = crop
        self.train = train

    def __len__(self):
        return len(self.indices)

    def _remap(self, mask: torch.Tensor) -> torch.Tensor:
        m = mask.long()
        out = torch.full_like(m, 255)
        fg = m >= 1
        out[fg] = m[fg] - 1
        return out

    def __getitem__(self, i):
        ex = self.ds[int(self.indices[i])]
        img = ex["image"].convert("RGB")
        msk = ex["annotation"]
        if self.train:
            scale = random.uniform(0.5, 2.0)
            new_size = (int(img.height * scale), int(img.width * scale))
            img = TF.resize(img, new_size, interpolation=T.InterpolationMode.BILINEAR)
            msk = TF.resize(msk, new_size, interpolation=T.InterpolationMode.NEAREST)
            pad_h = max(0, self.crop - img.height)
            pad_w = max(0, self.crop - img.width)
            if pad_h or pad_w:
                img = TF.pad(img, [0, 0, pad_w, pad_h], fill=0)
                msk = TF.pad(msk, [0, 0, pad_w, pad_h], fill=0)
            i0 = random.randint(0, img.height - self.crop)
            j0 = random.randint(0, img.width  - self.crop)
            img = TF.crop(img, i0, j0, self.crop, self.crop)
            msk = TF.crop(msk, i0, j0, self.crop, self.crop)
            if random.random() < 0.5:
                img = TF.hflip(img); msk = TF.hflip(msk)
            img = T.ColorJitter(0.4, 0.4, 0.4)(img)
        else:
            img = TF.resize(img, self.crop, interpolation=T.InterpolationMode.BILINEAR)
            msk = TF.resize(msk, self.crop, interpolation=T.InterpolationMode.NEAREST)
            img = TF.center_crop(img, self.crop)
            msk = TF.center_crop(msk, self.crop)
        img_t = TF.to_tensor(img)
        img_t = TF.normalize(img_t, IM_MEAN, IM_STD)
        msk_t = torch.as_tensor(np.array(msk), dtype=torch.long)
        msk_t = self._remap(msk_t)
        return img_t, msk_t


def build_loaders(cfg):
    _seed_everything(cfg.seed)
    ds = load_dataset("scene_parse_150", trust_remote_code=True)
    rng = np.random.RandomState(cfg.seed)
    train_idx = rng.choice(len(ds["train"]), size=cfg.subset_train, replace=False)
    val_idx   = rng.choice(len(ds["validation"]), size=cfg.subset_val, replace=False)
    train_ds = ADE20KSubset(ds["train"],      train_idx, cfg.crop, train=True)
    val_ds   = ADE20KSubset(ds["validation"], val_idx,   cfg.crop, train=False)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch, shuffle=True,  num_workers=2, drop_last=True, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch, shuffle=False, num_workers=2, drop_last=False, pin_memory=True)
    return train_loader, val_loader


## 7. Build model

In [ ]:
from transformers import SegformerForSemanticSegmentation

def build_model(cfg):
    model = SegformerForSemanticSegmentation.from_pretrained(
        cfg.pretrained,
        num_labels=cfg.num_classes,
        ignore_mismatched_sizes=True,
    )
    if cfg.variant == "ln":
        return model, []
    if cfg.variant == "dyt_full":
        return model, replace_ln_with_dyt(model, scope="all", alpha_init=cfg.dyt_alpha_init)
    if cfg.variant == "dyt_attn":
        return model, replace_ln_with_dyt(model, scope="attention", alpha_init=cfg.dyt_alpha_init)
    if cfg.variant == "dyt_ffn":
        return model, replace_ln_with_dyt(model, scope="ffn", alpha_init=cfg.dyt_alpha_init)
    raise ValueError(f"unknown variant: {cfg.variant}")


## 8. CPU smoke test

No GPU needed. Confirms DyT shapes for both `(B,N,C)` and `(B,C,H,W)`, plus replacement count > 0.

In [ ]:
def _cpu_smoke():
    cfg = Config(variant="dyt_full")
    m, replaced = build_model(cfg)
    m.eval()
    assert len(replaced) > 0, "no LN modules were replaced"
    x = torch.randn(1, 3, 64, 64)
    with torch.no_grad():
        out = m(pixel_values=x)
    assert out.logits.shape[0] == 1
    assert out.logits.shape[1] == cfg.num_classes
    # DyT direct shape tests
    d = DyT(32)
    assert d(torch.randn(2, 17, 32)).shape == (2, 17, 32)        # (B, N, C)
    assert d(torch.randn(2, 8, 4, 32)).shape == (2, 8, 4, 32)    # (B, ..., C) channel-last
    assert d(torch.randn(2, 32, 4, 8)).shape == (2, 32, 4, 8)    # (B, C, H, W)
    # ambiguous (B, C, H, C) must take the NCHW branch (not channel-last)
    out_amb = d(torch.randn(2, 32, 5, 32))
    assert out_amb.shape == (2, 32, 5, 32)
    print("CPU smoke OK: replaced", len(replaced), "LNs; logits shape", tuple(out.logits.shape))

_cpu_smoke()


## 9. GPU smoke test (one batch fwd/bwd, both variants)

In [ ]:
def _gpu_smoke():
    if not torch.cuda.is_available():
        print("skip GPU smoke: no CUDA"); return
    device = "cuda"
    cfg = Config(batch=2, crop=256, subset_train=8, subset_val=4)
    train_loader, _ = build_loaders(cfg)
    xb, yb = next(iter(train_loader))
    xb, yb = xb.to(device), yb.to(device)
    for variant in ["ln", "dyt_full"]:
        c = Config(**{**asdict(cfg), "variant": variant})
        m, _ = build_model(c)
        m = m.to(device)
        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            out = m(pixel_values=xb)
            logits = nn.functional.interpolate(out.logits, size=yb.shape[-2:], mode="bilinear", align_corners=False)
            loss = nn.functional.cross_entropy(logits, yb, ignore_index=cfg.ignore_index)
        loss.backward()
        assert torch.isfinite(loss), f"non-finite loss for {variant}"
        print(f"GPU smoke {variant}: loss={loss.item():.4f}")
        del m
        torch.cuda.empty_cache()

_gpu_smoke()


## 10. Train loop

- AdamW + polynomial LR (power 1.0) + linear warmup
- Mixed precision via `torch.cuda.amp.autocast` + `GradScaler` (fits T4 at crop=512)
- CE loss with `ignore_index=255`, logits upsampled to label resolution
- Eval every `cfg.eval_every`: confusion-matrix mIoU on val
- Logs JSONL to `final_paper/runs/<variant>/log.jsonl`
- Checkpoints: `best.pt` (updated first on improvement), then `last.pt` (always)
- Resumable: if `last.pt` exists, continues from there


In [ ]:
import json, time

def _poly_lr(base, step, total, warmup, power=1.0):
    if step < warmup:
        return base * step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return base * (1 - p) ** power


@torch.no_grad()
def run_eval(model, loader, num_classes: int, ignore_index: int, device: str, use_amp: bool):
    model.eval()
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long, device=device)
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp and device == "cuda"):
            out = model(pixel_values=xb).logits
            out = nn.functional.interpolate(out, size=yb.shape[-2:], mode="bilinear", align_corners=False)
        pred = out.argmax(1)
        valid = yb != ignore_index
        p = pred[valid]; t = yb[valid]
        idx = t * num_classes + p
        cm += torch.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    cm = cm.cpu()
    inter = cm.diag().float()
    union = cm.sum(0).float() + cm.sum(1).float() - inter
    iou = torch.where(union > 0, inter / union, torch.full_like(inter, float("nan")))
    miou = float(torch.nanmean(iou).item())
    return {"miou": miou, "iou_per_class": iou.tolist()}


def train_one(cfg, runs_dir):
    _seed_everything(cfg.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = cfg.use_amp and device == "cuda"

    run_dir = runs_dir / cfg.variant
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

    train_loader, val_loader = build_loaders(cfg)
    model, replaced = build_model(cfg)
    model = model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    start_iter = 0
    best_miou = -1.0
    last_path = run_dir / "last.pt"
    best_path = run_dir / "best.pt"
    if last_path.exists():
        ck = torch.load(last_path, map_location=device)
        model.load_state_dict(ck["model"]); optim.load_state_dict(ck["optim"])
        if "scaler" in ck and use_amp:
            scaler.load_state_dict(ck["scaler"])
        start_iter = ck["iter"]; best_miou = ck.get("best_miou", -1.0)
        print(f"resumed from iter {start_iter}, best_miou={best_miou:.4f}")

    log_path = run_dir / "log.jsonl"
    log_f = open(log_path, "a")
    train_iter = iter(train_loader)
    sat = SaturationTracker(model) if any(isinstance(m, DyT) for m in model.modules()) else None

    t0 = time.time()
    model.train()
    for step in range(start_iter, cfg.iters):
        try: xb, yb = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader); xb, yb = next(train_iter)
        xb, yb = xb.to(device), yb.to(device)
        lr = _poly_lr(cfg.lr, step, cfg.iters, cfg.warmup_iters)
        for g in optim.param_groups: g["lr"] = lr
        optim.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(pixel_values=xb).logits
            out = nn.functional.interpolate(out, size=yb.shape[-2:], mode="bilinear", align_corners=False)
            loss = nn.functional.cross_entropy(out, yb, ignore_index=cfg.ignore_index)
        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()

        if (step + 1) % 50 == 0:
            print(f"[{cfg.variant}] step {step+1}/{cfg.iters}  loss={loss.item():.4f}  lr={lr:.2e}  elapsed={time.time()-t0:.0f}s")
            log_f.write(json.dumps({"kind": "train", "step": step+1, "loss": float(loss.item()), "lr": lr}) + "\n"); log_f.flush()

        if (step + 1) % cfg.eval_every == 0 or (step + 1) == cfg.iters:
            metrics = run_eval(model, val_loader, cfg.num_classes, cfg.ignore_index, device, use_amp)
            sat_summary = sat.summary() if sat is not None else {}
            print(f"[{cfg.variant}] eval @ {step+1}: mIoU={metrics['miou']:.4f}")
            log_f.write(json.dumps({"kind": "eval", "step": step+1, "miou": metrics["miou"], "saturation": sat_summary}) + "\n"); log_f.flush()
            # update best FIRST so last.pt records the post-update best_miou
            if metrics["miou"] > best_miou:
                best_miou = metrics["miou"]
                torch.save({"model": model.state_dict(), "iter": step+1, "miou": best_miou}, best_path)
            torch.save({
                "model": model.state_dict(),
                "optim": optim.state_dict(),
                "scaler": scaler.state_dict() if use_amp else None,
                "iter": step+1,
                "best_miou": best_miou,
            }, last_path)
            model.train()

    log_f.close()
    if sat is not None: sat.close()
    summary = {"variant": cfg.variant, "iters": cfg.iters, "best_miou": best_miou,
               "num_replaced": len(replaced), "replaced_modules": replaced}
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    print("DONE", cfg.variant, "best_miou=", best_miou)
    return summary


## 11. Quick runs (500 iters)

**Run these first.** Both must produce: finite losses, non-empty `log.jsonl`, `best.pt`, `summary.json`, and at least one mIoU eval. Only after that, launch the 5000-iter runs in section 13.

In [ ]:
ln_quick = train_one(Config(variant="ln", iters=500, eval_every=250), RUNS_DIR)


In [ ]:
dyt_quick = train_one(Config(variant="dyt_full", iters=500, eval_every=250), RUNS_DIR)


## 12. Quick-run gate

Verifies the four quick-run preconditions before you hit the 5000-iter cells. If this assertion fails, **do not** proceed to section 13.

In [ ]:
def _gate(variant):
    rd = RUNS_DIR / variant
    log = (rd / "log.jsonl").read_text().strip().splitlines()
    assert len(log) > 0, f"{variant}: log.jsonl empty"
    evals = [json.loads(l) for l in log if json.loads(l)["kind"] == "eval"]
    assert len(evals) >= 1, f"{variant}: no eval rows"
    assert all(np.isfinite(json.loads(l)["loss"]) for l in log if json.loads(l)["kind"] == "train"), f"{variant}: non-finite training loss"
    assert (rd / "best.pt").exists(),    f"{variant}: best.pt missing"
    assert (rd / "summary.json").exists(), f"{variant}: summary.json missing"
    print(f"{variant} gate OK  best_miou={json.loads((rd/'summary.json').read_text())['best_miou']:.4f}")

_gate("ln")
_gate("dyt_full")
print("\nQuick runs PASSED. Safe to launch full 5000-iter runs.")


## 13. Full runs (5000 iters)

Only run after section 12 prints `Quick runs PASSED`.

**Note on resumability.** `train_one` resumes from `runs/<variant>/last.pt` if it exists. The cells below will therefore continue from the iter-500 quick-run checkpoints up to iter 5000 (NOT a fresh restart from iter 0). This is intended — it saves ~10 min per variant. If you want a clean fresh run instead, delete `runs/<variant>/last.pt` and `runs/<variant>/best.pt` before running these cells.

In [ ]:
ln_full = train_one(Config(variant="ln", iters=5000), RUNS_DIR)


In [ ]:
dyt_full_run = train_one(Config(variant="dyt_full", iters=5000), RUNS_DIR)


## 14. Diagnostics + figures

Reads `log.jsonl` from each run, writes PDFs to `final_paper/figures/`. `qualitative_grid` runs on GPU when available.

In [ ]:
import matplotlib.pyplot as plt

def _read_jsonl(p):
    return [json.loads(l) for l in open(p)]

def plot_curves(variants, runs_dir, figures_dir):
    fig_l, ax_l = plt.subplots(figsize=(5, 3))
    fig_m, ax_m = plt.subplots(figsize=(5, 3))
    for v in variants:
        log = _read_jsonl(runs_dir / v / "log.jsonl")
        ts = [r for r in log if r["kind"] == "train"]
        es = [r for r in log if r["kind"] == "eval"]
        ax_l.plot([r["step"] for r in ts], [r["loss"] for r in ts], label=v)
        ax_m.plot([r["step"] for r in es], [r["miou"] for r in es], marker="o", label=v)
    for ax, ttl, ylab in [(ax_l, "Training loss", "loss"), (ax_m, "Validation mIoU", "mIoU")]:
        ax.set_xlabel("iter"); ax.set_ylabel(ylab); ax.set_title(ttl); ax.legend(); ax.grid(alpha=0.3)
    fig_l.tight_layout(); fig_m.tight_layout()
    fig_l.savefig(figures_dir / "loss_curve.pdf")
    fig_m.savefig(figures_dir / "miou_curve.pdf")
    print("saved loss_curve.pdf, miou_curve.pdf")


def plot_saturation(variants, runs_dir, figures_dir):
    fig, ax = plt.subplots(figsize=(6, 3))
    for v in variants:
        log = _read_jsonl(runs_dir / v / "log.jsonl")
        evs = [r for r in log if r["kind"] == "eval" and r.get("saturation")]
        if not evs: continue
        last = evs[-1]["saturation"]
        names = list(last.keys()); vals = [last[n] for n in names]
        ax.plot(range(len(vals)), vals, marker=".", label=v)
    ax.set_xlabel("DyT layer index (depth-ordered)")
    ax.set_ylabel("frac |tanh(alpha*x)| > 0.95")
    ax.set_title("DyT saturation by layer")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(figures_dir / "saturation.pdf")
    print("saved saturation.pdf")


def qualitative_grid(variants, runs_dir, figures_dir, n_images=6):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _, val_loader = build_loaders(Config(subset_val=n_images, batch=n_images))
    xb, yb = next(iter(val_loader))
    rows = [("input", xb.cpu()), ("gt", yb.cpu())]
    for v in variants:
        c = Config(variant=v)
        m, _ = build_model(c)
        ck = torch.load(runs_dir / v / "best.pt", map_location=device)
        m.load_state_dict(ck["model"]); m.to(device).eval()
        with torch.no_grad():
            xb_d = xb.to(device)
            out = m(pixel_values=xb_d).logits
            out = nn.functional.interpolate(out, size=yb.shape[-2:], mode="bilinear", align_corners=False)
            pred = out.argmax(1).cpu()
        rows.append((v, pred))
        del m
        if device == "cuda": torch.cuda.empty_cache()
    fig, axes = plt.subplots(len(rows), n_images, figsize=(2 * n_images, 2 * len(rows)))
    for r, (label, batch) in enumerate(rows):
        for c_ in range(n_images):
            ax = axes[r, c_] if len(rows) > 1 else axes[c_]
            if label == "input":
                im = batch[c_].permute(1, 2, 0).numpy()
                im = (im * np.array(IM_STD) + np.array(IM_MEAN)).clip(0, 1)
                ax.imshow(im)
            else:
                ax.imshow(batch[c_].numpy(), cmap="tab20")
            if c_ == 0: ax.set_ylabel(label, fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout(); fig.savefig(figures_dir / "qualitative.pdf")
    print("saved qualitative.pdf")


variants = ["ln", "dyt_full"]
plot_curves(variants, RUNS_DIR, FIGURES_DIR)
plot_saturation(["dyt_full"], RUNS_DIR, FIGURES_DIR)
qualitative_grid(variants, RUNS_DIR, FIGURES_DIR)


## 15. Summary table

In [ ]:
rows = []
for v in ["ln", "dyt_full"]:
    p = RUNS_DIR / v / "summary.json"
    if p.exists():
        rows.append(json.loads(p.read_text()))
print(json.dumps(rows, indent=2))
(FIGURES_DIR / "table_main.json").write_text(json.dumps(rows, indent=2))
